In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

# Part 1

## 1.2 Data Inspection and Validation

### Load in Data

In [2]:
DATA_DIR = Path("../data")

df = pd.read_csv(DATA_DIR / "raw" / "housing.csv")

### Inspect

In [3]:
print(df.head())
print(df.shape)

   id  size_sqft  bedrooms  age_years  price_thousands
0   1     1331.8         3       13.2           261.27
1   2     3109.3         6       29.0           582.58
2   3     1219.4         3       39.9           255.72
3   4     1868.1         4       26.6           374.11
4   5     1690.4         3       23.2           356.49
(130, 5)


In [4]:
print(df.dtypes)
print(df.isna().sum())
print(df["id"].is_unique)

id                   int64
size_sqft          float64
bedrooms             int64
age_years          float64
price_thousands    float64
dtype: object
id                 0
size_sqft          0
bedrooms           0
age_years          0
price_thousands    0
dtype: int64
True


# Part 2

In [5]:
predictors = df[["size_sqft", "bedrooms", "age_years"]].to_numpy()
y = df["price_thousands"].to_numpy()

### Standardize the Predictors

In [6]:
means = predictors.mean(axis=0)
sds = predictors.std(axis=0)

Z = (predictors- means) / sds

print(Z.mean(axis=0))
print(Z.std(axis=0))

[-3.34774943e-16  9.56499837e-17  1.43474975e-16]
[1. 1. 1.]


### Construct the Design Matrix

In [7]:
X = np.column_stack([np.ones(len(Z)), Z])

In [8]:
predictors.shape

(130, 3)

In [9]:
Z.shape

(130, 3)

In [10]:
X.shape

(130, 4)

In [12]:
y.shape

(130,)

In [ ]:
np.linalg.matrix_rank(X)

np.int64(4)

# Part 3

### Fit the Model

In [14]:
beta_hat, *_ = np.linalg.lstsq(X, y, rcond = None)
beta_hat

array([378.69146154,  66.79748572,  13.69239326, -17.92866377])

### Calculate Fitted Values and Residuals

In [15]:
# Fitted Values
fitted = X @ beta_hat

# Residuals
residuals = y- fitted

In [16]:
residuals.mean()

np.float64(5.24708481792074e-15)

In [17]:
rmse = np.sqrt(np.mean(residuals ** 2))
rmse

np.float64(28.818978213959117)

# Part 4

## QR Decomposition

In [18]:
Q, R = np.linalg.qr(X)

print(Q.shape)
print(R.shape)

(130, 4)
(4, 4)


In [24]:
np.allclose(Q @ R, X)

True

In [25]:
np.allclose(Q.T @ Q, np.eye(Q.shape[1]))

True

## SVD

In [27]:
U, s, Vt = np.linalg.svd(X, full_matrices=False)

In [28]:
print(U.shape)
print(s.shape)
print(Vt.shape)

(130, 4)
(4,)
(4, 4)


In [33]:
print(s)

[15.60230122 11.40175425 10.66724547  5.72521361]


### Reconstruct X

In [34]:
X_reconstructed = (U @ np.diag(s) @ Vt)

In [35]:
np.allclose(X_reconstructed, X)

True

## Rank and Conditioning

In [36]:
print(np.linalg.matrix_rank(X))
print(np.linalg.cond(X))

4
2.725191106858789


In [38]:
X_raw = np.column_stack([np.ones(len(predictors)), predictors])
np.linalg.cond(X_raw)

np.float64(10733.342974769754)